# BanglaBERT Transformer Baseline - All Binary Datasets (Loop Version)

This notebook trains and evaluates **BanglaBERT** on all three binary datasets in one run:

- `ben_sarc_binary`
- `banglasarc_binary`
- `banglasarc3_binary`

It avoids manual dataset switching and uses `trainer.predict(...)` for final evaluation to avoid the callback issue you saw with `trainer.evaluate(...)`.

In [1]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Torch version:", torch.__version__)
try:
    import transformers
    print("Transformers version:", transformers.__version__)
except Exception as e:
    print("Could not read transformers version:", e)

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("Detected device:", device)

Torch version: 2.11.0
Transformers version: 5.3.0
Detected device: mps


In [3]:
# Paths
ROOT = Path("..")
SPLITS = ROOT / "01_data" / "interim" / "splits"
CHECKPOINTS = ROOT / "03_models" / "checkpoints"
TABLES = ROOT / "04_outputs" / "tables"

CHECKPOINTS.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

# Model
MODEL_NAME = "csebuetnlp/banglabert"
MAX_LENGTH = 128
BATCH_SIZE = 8
EPOCHS = 2
LR = 2e-5
WEIGHT_DECAY = 0.01

DATASET_NAMES = [
    "ben_sarc_binary",
    "banglasarc_binary",
    "banglasarc3_binary",
]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Splits path:", SPLITS.resolve())
print("Tables path:", TABLES.resolve())

Splits path: /Users/sefayet/Desktop/Github/Machine_Learning-Deep_Learning-Courses-and-Paper-Publish/Thesis_Papers/Sarcasm_detection/01_data/interim/splits
Tables path: /Users/sefayet/Desktop/Github/Machine_Learning-Deep_Learning-Courses-and-Paper-Publish/Thesis_Papers/Sarcasm_detection/04_outputs/tables


In [4]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )

    return {
        "accuracy": acc,
        "precision_binary": p_bin,
        "recall_binary": r_bin,
        "f1_binary": f1_bin,
        "macro_f1": f1_macro,
    }

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

def prepare_hf_datasets(train_df, val_df, test_df, label_col="label_binary"):
    train_df = train_df[["text", label_col]].rename(columns={label_col: "label"})
    val_df = val_df[["text", label_col]].rename(columns={label_col: "label"})
    test_df = test_df[["text", label_col]].rename(columns={label_col: "label"})

    train_ds = Dataset.from_pandas(train_df, preserve_index=False)
    val_ds = Dataset.from_pandas(val_df, preserve_index=False)
    test_ds = Dataset.from_pandas(test_df, preserve_index=False)

    train_ds = train_ds.map(tokenize_batch, batched=True)
    val_ds = val_ds.map(tokenize_batch, batched=True)
    test_ds = test_ds.map(tokenize_batch, batched=True)

    train_ds = train_ds.remove_columns(["text"])
    val_ds = val_ds.remove_columns(["text"])
    test_ds = test_ds.remove_columns(["text"])

    train_ds.set_format("torch")
    val_ds.set_format("torch")
    test_ds.set_format("torch")

    return train_df, val_df, test_df, train_ds, val_ds, test_ds

def predict_metrics(trainer, df_for_labels, ds):
    output = trainer.predict(ds)
    preds = np.argmax(output.predictions, axis=-1)
    labels = np.array(df_for_labels["label"])

    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    cm = confusion_matrix(labels, preds)

    metrics = {
        "accuracy": float(acc),
        "precision_binary": float(p_bin),
        "recall_binary": float(r_bin),
        "f1_binary": float(f1_bin),
        "macro_f1": float(f1_macro),
    }
    return metrics, cm.tolist()

In [5]:
all_rows = []
all_confusions = {}

for dataset_name in DATASET_NAMES:
    print("\n" + "=" * 80)
    print("RUNNING DATASET:", dataset_name)
    print("=" * 80)

    train_path = SPLITS / f"{dataset_name}_train.csv"
    val_path = SPLITS / f"{dataset_name}_val.csv"
    test_path = SPLITS / f"{dataset_name}_test.csv"

    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)
    test_df = pd.read_csv(test_path)

    print("Loaded shapes:")
    print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)

    train_df2, val_df2, test_df2, train_ds, val_ds, test_ds = prepare_hf_datasets(
        train_df, val_df, test_df, label_col="label_binary"
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    training_args = TrainingArguments(
    output_dir=str(CHECKPOINTS / f"banglabert_{dataset_name}"),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    train_output = trainer.train()
    print("Training completed.")
    print(train_output)

    # Final metrics via predict() to avoid the callback-state problem you saw with evaluate()
    val_metrics, val_cm = predict_metrics(trainer, val_df2, val_ds)
    test_metrics, test_cm = predict_metrics(trainer, test_df2, test_ds)

    print("\nValidation metrics:", val_metrics)
    print("Validation confusion matrix:", val_cm)
    print("\nTest metrics:", test_metrics)
    print("Test confusion matrix:", test_cm)

    all_confusions[dataset_name] = {
        "validation": val_cm,
        "test": test_cm,
    }

    all_rows.append({
        "model": "banglabert",
        "dataset": dataset_name,
        "split": "validation",
        "accuracy": val_metrics["accuracy"],
        "precision_binary": val_metrics["precision_binary"],
        "recall_binary": val_metrics["recall_binary"],
        "f1_binary": val_metrics["f1_binary"],
        "macro_f1": val_metrics["macro_f1"],
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "max_length": MAX_LENGTH,
        "seed": SEED,
    })

    all_rows.append({
        "model": "banglabert",
        "dataset": dataset_name,
        "split": "test",
        "accuracy": test_metrics["accuracy"],
        "precision_binary": test_metrics["precision_binary"],
        "recall_binary": test_metrics["recall_binary"],
        "f1_binary": test_metrics["f1_binary"],
        "macro_f1": test_metrics["macro_f1"],
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "max_length": MAX_LENGTH,
        "seed": SEED,
    })

    per_dataset_df = pd.DataFrame([row for row in all_rows if row["dataset"] == dataset_name])
    per_dataset_csv = TABLES / f"banglabert_{dataset_name}_results.csv"
    per_dataset_df.to_csv(per_dataset_csv, index=False)

    per_dataset_cm_json = TABLES / f"banglabert_{dataset_name}_confusion_matrices.json"
    with open(per_dataset_cm_json, "w", encoding="utf-8") as f:
        json.dump(all_confusions[dataset_name], f, ensure_ascii=False, indent=2)

    print("\nSaved:")
    print("-", per_dataset_csv)
    print("-", per_dataset_cm_json)


RUNNING DATASET: ben_sarc_binary
Loaded shapes:
Train: (20508, 4) Val: (2564, 4) Test: (2564, 4)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 49031.44it/s]
ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
summary_df = pd.DataFrame(all_rows).sort_values(["dataset", "split"]).reset_index(drop=True)
summary_df

In [ ]:
summary_csv = TABLES / "banglabert_binary_summary.csv"
summary_df.to_csv(summary_csv, index=False)

all_cm_json = TABLES / "banglabert_binary_all_confusion_matrices.json"
with open(all_cm_json, "w", encoding="utf-8") as f:
    json.dump(all_confusions, f, ensure_ascii=False, indent=2)

print("Saved summary CSV:", summary_csv)
print("Saved all confusion matrices JSON:", all_cm_json)
print("\nSaved table files:")
for p in sorted(TABLES.glob("banglabert_*")):
    print("-", p.name)